# CLaRa — Fine-Tuning & Evaluation (Kaggle T4)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Checkpoint:** `tokiggle/clara-7b-e2e-4q` (Kaggle dataset)

---

## Overview

This notebook fine-tunes the Apple CLaRa-7B-E2E pretrained checkpoint using **Apple's own `modeling_clara.py`** (4-bit NF4), then evaluates both pretrained and fine-tuned models.

### Fine-Tuning Strategy

| Component | Status | Rationale |
|-----------|--------|-----------|
| `encoder_adapter` | **Frozen** | Compressor was trained by Apple on 2M docs (Stage I) |
| `query_reasoner_adapter` | **Trained** | Adapts query reasoning to target domain |
| `decoder_adapter` | **Trained** | Adapts answer generation to target domain |

### Notebook Structure

```
Section 0 — Environment Setup
Section 1 — Fine-tune on SQuAD + Evaluate
Section 2 — Fine-tune on TriviaQA + Evaluate  
Section 3 — Results Summary
```

---
## Section 0 — Environment Setup

**Run once.** Clone the repo and install dependencies, then **restart the kernel**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Restart kernel.
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"

print("[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", REPO_ROOT], check=True)
subprocess.run(["git", "clone", "-b", "feature-tuyen2", REPO_URL, REPO_ROOT], check=True)

print("[2/3] Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     f"{REPO_ROOT}/requirements.txt", "-q"],
    check=True,
)

print("[3/3] Running setup script...")
subprocess.run([sys.executable, f"{REPO_ROOT}/setup_env.py"], check=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

In [ ]:
import shutil, os

# Kill the hidden HF code cache to force fresh model code loading
cache_path = "/root/.cache/huggingface/modules/transformers_modules"
if os.path.exists(cache_path):
    print("Nuking stale code cache...")
    shutil.rmtree(cache_path)
    print("Cache destroyed.")
else:
    print("Cache was already empty.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-B  │  Working directory & path configuration
# Run this cell FIRST after every kernel restart.
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), f"Repository not found at {REPO_ROOT}. Run Cell 0-A first."

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-C  │  GPU / VRAM diagnostics
# ═══════════════════════════════════════════════════════════════════════════════

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

if vram_gb < 14:
    print("\n Warning: Less than 14 GB VRAM.")
else:
    print("\n VRAM sufficient for fine-tuning (batch_size=1, grad_accum=8).")

---
## Section 1 — Fine-Tune on SQuAD + Evaluate

Fine-tunes `query_reasoner_adapter` + `decoder_adapter` on SQuAD train split, then evaluates.

| Setting | Value |
|---------|-------|
| Train samples | 2,000 |
| Val samples | 200 |
| LR | 5e-6 (cosine decay) |
| Grad accumulation | 8 |
| Epochs | 1 |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1-A  │  Fine-tune on SQuAD
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"
FT_SQUAD_DIR = "/kaggle/working/clara-ft-squad"

print("╔" + "═" * 60 + "╗")
print("║  Fine-Tune CLaRa on SQuAD (Apple Native Pipeline)           ║")
print("╠" + "═" * 60 + "╣")
print("║  Adapters trained : query_reasoner + decoder (LoRA)         ║")
print("║  Encoder adapter  : FROZEN                                  ║")
print("║  Train/Val        : 2000/200  |  LR: 5e-6  |  Epochs: 1    ║")
print("║  Batch size=1, grad_accum=8 (effective batch=8)             ║")
print("╚" + "═" * 60 + "╝")

ft_env = os.environ.copy()
ft_env.update({
    "CLARA_CKPT_PATH"    : APPLE_CKPT,
    "CLARA_DATASET"      : "squad",
    "CLARA_N_TRAIN"      : "2000",
    "CLARA_N_VAL"        : "200",
    "CLARA_FT_LR"        : "5e-6",
    "CLARA_FT_EPOCHS"    : "1",
    "CLARA_FT_GRAD_ACC"  : "8",
    "CLARA_FT_MAX_DEC_LEN": "512",
    "CLARA_OUTPUT_DIR"   : FT_SQUAD_DIR,
    "CLARA_MODEL_VERSION": "FT_SQuAD",
})

subprocess.run(
    ["python", "-m", "scripts.finetune_apple"],
    env=ft_env, check=True,
)

print(f"\n✅ Fine-tuning complete. Checkpoint: {FT_SQUAD_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1-B  │  Evaluate fine-tuned model on SQuAD
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess, gc, torch

# Free VRAM from training before loading for eval
gc.collect()
torch.cuda.empty_cache()

FT_SQUAD_DIR = "/kaggle/working/clara-ft-squad"

print("╔" + "═" * 60 + "╗")
print("║  Evaluate Fine-Tuned Model on SQuAD                         ║")
print("╠" + "═" * 60 + "╣")
print("║  Checkpoint : Apple init → fine-tuned on SQuAD              ║")
print("║  Eval mode  : oracle  |  Metrics: EM + F1                   ║")
print("╚" + "═" * 60 + "╝")

eval_env = os.environ.copy()
eval_env.update({
    "CLARA_CKPT_PATH"     : FT_SQUAD_DIR,
    "CLARA_DATASET"       : "squad",
    "CLARA_EVAL_MODE"     : "oracle",
    "CLARA_EVAL_BS"       : "1",
    "CLARA_N_VAL"         : "500",
    "CLARA_MODEL_VERSION" : "ModelB_FineTuned_SQuAD",
    "CLARA_MAX_NEW_TOKENS": "32",
})

subprocess.run(
    ["python", "-m", "scripts.evaluate_apple"],
    env=eval_env, check=True,
)

print("\n✅ SQuAD evaluation complete. Results saved to results/eval_scores.csv")

---
## Section 2 — Fine-Tune on TriviaQA + Evaluate

Same procedure as Section 1, but on TriviaQA (`rc.nocontext` variant to avoid OOM).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2-A  │  Fine-tune on TriviaQA
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess, gc, torch

gc.collect()
torch.cuda.empty_cache()

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"
FT_TRIVIAQA_DIR = "/kaggle/working/clara-ft-triviaqa"

print("╔" + "═" * 60 + "╗")
print("║  Fine-Tune CLaRa on TriviaQA (Apple Native Pipeline)       ║")
print("╠" + "═" * 60 + "╣")
print("║  Adapters trained : query_reasoner + decoder (LoRA)         ║")
print("║  Encoder adapter  : FROZEN                                  ║")
print("║  Train/Val        : 2000/200  |  LR: 5e-6  |  Epochs: 1    ║")
print("║  Dataset variant  : rc.nocontext (avoids 10GB download)     ║")
print("╚" + "═" * 60 + "╝")

ft_env = os.environ.copy()
ft_env.update({
    "CLARA_CKPT_PATH"    : APPLE_CKPT,
    "CLARA_DATASET"      : "triviaqa",
    "CLARA_N_TRAIN"      : "2000",
    "CLARA_N_VAL"        : "200",
    "CLARA_FT_LR"        : "5e-6",
    "CLARA_FT_EPOCHS"    : "1",
    "CLARA_FT_GRAD_ACC"  : "8",
    "CLARA_FT_MAX_DEC_LEN": "512",
    "CLARA_OUTPUT_DIR"   : FT_TRIVIAQA_DIR,
    "CLARA_MODEL_VERSION": "FT_TriviaQA",
})

subprocess.run(
    ["python", "-m", "scripts.finetune_apple"],
    env=ft_env, check=True,
)

print(f"\n✅ Fine-tuning complete. Checkpoint: {FT_TRIVIAQA_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2-B  │  Evaluate fine-tuned model on TriviaQA
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess, gc, torch

gc.collect()
torch.cuda.empty_cache()

FT_TRIVIAQA_DIR = "/kaggle/working/clara-ft-triviaqa"

print("╔" + "═" * 60 + "╗")
print("║  Evaluate Fine-Tuned Model on TriviaQA                     ║")
print("╠" + "═" * 60 + "╣")
print("║  Checkpoint : Apple init → fine-tuned on TriviaQA           ║")
print("║  Eval mode  : oracle  |  Metrics: EM + F1                   ║")
print("╚" + "═" * 60 + "╝")

eval_env = os.environ.copy()
eval_env.update({
    "CLARA_CKPT_PATH"     : FT_TRIVIAQA_DIR,
    "CLARA_DATASET"       : "triviaqa",
    "CLARA_EVAL_MODE"     : "oracle",
    "CLARA_EVAL_BS"       : "1",
    "CLARA_N_VAL"         : "500",
    "CLARA_MODEL_VERSION" : "ModelB_FineTuned_TriviaQA",
    "CLARA_MAX_NEW_TOKENS": "32",
})

subprocess.run(
    ["python", "-m", "scripts.evaluate_apple"],
    env=eval_env, check=True,
)

print("\n✅ TriviaQA evaluation complete. Results saved to results/eval_scores.csv")

---
## Section 3 — Results Summary

Aggregates all evaluation results and displays a comparison table.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3  │  Results summary — all evaluation runs
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}. Run evaluation cells first.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EXPERIMENT RESULTS SUMMARY")
    print("═" * 80)
    print()

    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()

    display_df = display_df.sort_values(['Dataset', 'Model_Version'])

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))

    print()
    print("─" * 80)
    print("PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):")
    print("  NQ        : EM=63.29%  F1=71.54%")
    print("  HotpotQA  : EM=57.54%  F1=71.17%")
    print("  (Instruction-tuned init, Normal setting)")
    print("─" * 80)
    print()
    print("NOTE: Fine-tuned results should ideally improve over pretrained baseline.")
    print("If they don't, consider: more training samples, lower LR, or more epochs.")

---
## Experimental Notes

### Fine-Tuning Strategy

| Setting | Value | Rationale |
|---------|-------|-----------|
| Frozen adapter | `encoder_adapter` | SCP pretraining by Apple is high quality |
| Trained adapters | `query_reasoner_adapter` + `decoder_adapter` | Stage II adapters |
| Batch size | 1 | T4 VRAM constraint |
| Grad accumulation | 8 | Effective batch size of 8 |
| LR | 5e-6 | Paper Table 10 (Stage II LR) |
| Train samples | 2,000 | Kaggle runtime constraint |

### Citation

```bibtex
@article{he2026clara,
  title   = {CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning},
  author  = {He, Jie and Bai, Richard He and Williamson, Sinead and Pan, Jeff Z. 
             and Jaitly, Navdeep and Zhang, Yizhe},
  journal = {arXiv preprint arXiv:2511.18659},
  year    = {2026}
}
```